In [2]:
import torch
import torch.nn as nn
import numpy as np
import librosa
from torchvision.models import resnet18, ResNet18_Weights

# -------------------
# CONFIG
# -------------------
SAMPLE_RATE = 16000
DURATION = 5
SAMPLES = SAMPLE_RATE * DURATION

N_MELS = 128
N_FFT = 1024
HOP_LENGTH = 256

CHECKPOINT_PATH = "best_model.pth"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------
# Load Model
# -------------------
def load_model(num_classes):
    model = resnet18(weights=ResNet18_Weights.DEFAULT)

    # Convert first layer to accept 1 channel instead of 3
    w = model.conv1.weight.data
    new_conv = nn.Conv2d(
        1, w.shape[0], kernel_size=7, stride=2, padding=3, bias=False
    )
    new_conv.weight.data = w.mean(dim=1, keepdim=True)
    model.conv1 = new_conv

    # Replace final FC layer
    model.fc = nn.Linear(model.fc.in_features, num_classes)

    return model

# -------------------
# Preprocess Audio
# -------------------
def preprocess_audio(audio_path):
    y, sr = librosa.load(audio_path, sr=SAMPLE_RATE)

    # pad or trim to exactly 5 seconds
    if len(y) < SAMPLES:
        y = np.pad(y, (0, SAMPLES - len(y)), mode='constant')
    else:
        y = y[:SAMPLES]

    # mel spectrogram
    mel = librosa.feature.melspectrogram(
        y=y, sr=SAMPLE_RATE, n_fft=N_FFT,
        hop_length=HOP_LENGTH, n_mels=N_MELS
    )

    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-9)

    # shape: (1, 1, 128, time)
    mel_tensor = torch.tensor(mel_db).unsqueeze(0).unsqueeze(0).float()
    return mel_tensor

# -------------------
# Predict One File
# -------------------
def predict_audio(file_path):
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

    class_to_idx = checkpoint["class_to_idx"]
    idx_to_class = {v: k for k, v in class_to_idx.items()}
    num_classes = len(class_to_idx)

    model = load_model(num_classes)
    model.load_state_dict(checkpoint["model"])
    model.to(DEVICE)
    model.eval()

    mel = preprocess_audio(file_path).to(DEVICE)

    with torch.no_grad():
        output = model(mel)
        probs = torch.softmax(output, dim=1)[0]
        pred_idx = torch.argmax(probs).item()

    pred_class = idx_to_class[pred_idx]
    confidence = float(probs[pred_idx])

    return pred_class, confidence

# -------------------
# MAIN (UPDATE FILE PATH HERE ONLY)
# -------------------
if __name__ == "__main__":

    # ⭐⭐ CHANGE THIS LINE ONLY ⭐⭐
    file_path = "voice/elephant/test1.wav"
    # Example: file_path = "my_audio.wav"
    # Example: file_path = "/Users/yuvraj/Desktop/sound.wav"

    pred_class, conf = predict_audio(file_path)

    print("=====================================")
    print(f"File: {file_path}")
    print(f"Predicted Class: {pred_class}")
    print(f"Confidence: {conf*100:.2f}%")
    print("=====================================")


/var/folders/rr/d0gz2wj51xg3h4b_s1cxz0bc0000gn/T/ipykernel_49820/2462505293.py:44: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(audio_path, sr=SAMPLE_RATE)


FileNotFoundError: [Errno 2] No such file or directory: 'voice/elephant/test1.wav'